# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list top-level record sets and their corresponding fields using the `@id`. All further references to records, record sets, and fields are done by their `@id`.

In [ ]:
# Identify all record sets in the metadata by their @id

record_sets = getattr(metadata, 'recordSet', [])
if not isinstance(record_sets, list):
    record_sets = [record_sets]  # ensure it's a list

print('Available record sets (referenced by @id):')
for rs in record_sets:
    if hasattr(rs, '@id'):
        print(f"  - {rs['@id']}")
    elif isinstance(rs, dict) and '@id' in rs:
        print(f"  - {rs['@id']}")
    elif isinstance(rs, str):
        print(f"  - {rs}")

record_set_ids = []
for rs in record_sets:
    if hasattr(rs, '@id'):
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)

# If there are no record sets found, display a message
if not record_set_ids:
    print('No record sets found in metadata, fetching from the Croissant schema directly as a fallback for demo.')

    # Try to fallback: scan all dataset objects looking for cr:RecordSet or objects with '@type' == 'RecordSet'
    # Load the raw json for exploration
    import requests
    raw = requests.get(url).json()
    found_rs = []
    for elem in raw.get('@graph', []):
        if ('@type' in elem and ('RecordSet' in elem['@type'] or elem['@type']=='cr:RecordSet')):
            found_rs.append(elem['@id'])
    record_set_ids = found_rs
    for _id in record_set_ids:
        print(f"  - {_id}")

if not record_set_ids:
    print('No record sets are defined in this Croissant schema! Please check the dataset or schema.')

Let's review one of the record sets and inspect its available fields by their `@id`. We'll use the first detected record set as an example.

In [ ]:
# List all fields for one record set, using @id

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Fields for record set {example_record_set_id}:")

    # Try fetching the record set object from the schema
    import requests
    schema = requests.get(url).json()

    # Look for the record set definition
    rs_obj = None
    for obj in schema.get('@graph', []):
        if obj.get('@id') == example_record_set_id:
            rs_obj = obj
            break
    field_ids = []
    if rs_obj is not None and 'field' in rs_obj:
        fields = rs_obj['field']
        if isinstance(fields, dict) and '@id' in fields:
            field_ids = [fields['@id']]
        elif isinstance(fields, list):
            for f in fields:
                if isinstance(f, dict) and '@id' in f:
                    field_ids.append(f['@id'])
                elif isinstance(f, str):
                    field_ids.append(f)
    else:
        print(f"No fields found for record set {example_record_set_id}.")
    for f in field_ids:
        print(f"  - {f}")
else:
    print('No record sets found, cannot show fields.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For multiple record sets, repeat as needed. We'll extract one record set as an example.

In [ ]:
# Extract data from each record set and load as DataFrames (using @id)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load data for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Choose a numeric field to filter and normalize. Please refer to the printed column names above to pick the best field.

In [ ]:
# Change these IDs to match the relevant column @id found above for real analysis.
import numpy as np

if record_set_ids:
    chosen_record_set = record_set_ids[0]  # Use first as example
    df = dataframes[chosen_record_set]
    print(f"Working with record set: {chosen_record_set}")

    # Try to automatically select a likely numeric field
    possible_numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    if possible_numeric:
        numeric_field = possible_numeric[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        # fallback: pick first column
        numeric_field = df.columns[0] if len(df.columns) else None
        print("No numeric dtype found, using first column (may not be suitable):", numeric_field)

    if numeric_field:
        df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanpercentile(df_numeric, 75) if np.nanmax(df_numeric) else 10
        filtered_df = df[df_numeric > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (df_numeric - df_numeric.mean()) / df_numeric.std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field if one exists
        possible_group = [col for col in df.columns if df[col].dtype==object and col!=numeric_field]
        group_field = possible_group[0] if possible_group else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing means of numeric fields):")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
    else:
        print("No fields found for EDA.")
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Example plot: histogram of the first numeric field
if record_set_ids and numeric_field:
    df = dataframes[chosen_record_set]
    df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
    plt.figure(figsize=(8,4))
    plt.hist(df_numeric.dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Example: boxplot grouped by grouping field (if exists)
if record_set_ids and numeric_field and group_field:
    df2 = df[[numeric_field, group_field]].copy()
    df2[numeric_field] = pd.to_numeric(df2[numeric_field], errors='coerce')
    df2 = df2.dropna()
    if not df2.empty:
        plt.figure(figsize=(10,5))
        df2.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle("")
        plt.ylabel(numeric_field)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.